I just wanna make it so that you don't need a target site to predict insertion LE/RE

In [ ]:
import sys
sys.path.append("../scripts")
import predict_recombination_products as pr
import numpy as np
from Bio import SeqIO
import os
pr

class Insertion(pr.BridgeRNAReaction):
    """
    A workaround for the BridgeRNAReaction class to allow detection of insertion
    sequence ends without providing the sequence to insert into.
    """
    def __init__(self, brna_path, other_sequence_paths, 
                 annotate_split_features=True, verify_unique_target=True):
        self.brna_path = brna_path
        self.other_sequence_paths = other_sequence_paths
        self.annotate_split_features = annotate_split_features
        self.verify_unique_target = verify_unique_target
        self._process_brna()
        self._identify_donor()

    def _identify_donor(self):
        # Read other sequence records
        sequence_paths = [self.brna_path]
        # Store all sequence records in additional sequence files
        other_sequence_records = []
        for filepath in self.other_sequence_paths:
            records = SeqIO.parse(filepath, "genbank")
            for record in records:
                    other_sequence_records.append(record)
                    sequence_paths.append(filepath)
        # other_sequence_records = [
        #     SeqIO.read(file, "genbank") for file in self.other_sequence_paths
        #     ]
        sequence_records = [self.brna_plasmid_record] + other_sequence_records
        seqs = [str(record.seq) for record in sequence_records]
        # Search all sequence records for target and donor sites
        target_matches = [
            self.find_matches(self.target_site, i) for i in seqs]
        donor_matches = [
            self.find_matches(self.donor_site, i) for i in seqs]

        target_match_contigs = [bool(i) for i in target_matches]
        target_index = np.where(target_match_contigs)[0].item() if any(target_match_contigs) else None
        donor_index = np.where([bool(i) for i in donor_matches])[0].item()
        assert (target_index != donor_index), "Target and Donor found on same contig, so not an insertion!"
        self.donor_strand, self.donor_start, self.donor_end = donor_matches[donor_index][0]
        self.mode = "insertion"
        # Store separate donor record for insertions
        self.donor_file = os.path.splitext(os.path.basename(sequence_paths[donor_index]))[0]
        print(f"Found donor sequence: {self.donor_file}\n\tstrand: {self.donor_strand}")
        self.donor_contig = sequence_records[donor_index]
        self._find_donor_site()

    def _find_donor_site(self):
        # Store donor site
        if self.donor_strand == 1:
            self.donor_split_site = self.donor_end - len(self.rt)
        else:
            self.donor_split_site = self.donor_start + len(self.rt)




brna_path = "/groups/cress/projects/efactors/jack/insertion_mapping/data/host_range/plasmids/4PVXT6_1_gg549_1_pJRP1063.gbk"
other_sequence_paths = ""

Insertion(brna_path, other_sequence_paths)
# pr.BridgeRNAReaction(brna_path, other_sequence_paths, verify_unique_target=False)

Found bRNA on (+) strand, starting at 2664: AGTGCAGAGAAAATCGGCCAGTTTTCTCTGCCTGCAGTCCGCATGCCGTAGCCGCGGTTGGGTTCTAACCTGTGTATTACATTTATGCAGCGGACTGCCTTTCTCCCAAAGTGATAAACCGGACAGTATGATGGACCGGTTTTCCCGGTAATCCGTATTTACAACCGTGGTTTCACT
LTG: AGCCGCGGT
RTG: GTATTAC
Target site: AGCCGCGGTAATAC
    LT: AGCCGCG target core: GT RT: AATAC
LDG: ACAGTATG
RDG1: ACAAC
RDG2: AT
Donor site: ACAGTATGTTGTAT
    LD: ACAGTAT donor core: GT RD: TGTAT
Using:	Target site: AGCCGCGGTAATAC
	Donor site: ACAGTATGTTGTAT
Found donor sequence: 4PVXT6_1_gg549_1_pJRP1063
	strand: 1


AttributeError: 'Insertion' object has no attribute 'target_strand'

In [17]:
[True, True] == [True, False]

False